# 02 高斯消元与主元选择

高斯消元通过初等行变换把线性系统化为上三角系统，再用回代求解。本节先展示不选主元的教学实现，再加入部分选主元，并用一个小主元例子说明主元选择的必要性。


In [ ]:
import pathlib
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch06_direct_linear_systems"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import gaussian_elimination, gaussian_elimination_partial_pivoting
from py_sc.direct_linear import relative_forward_error, relative_residual


## 前向消元

第 $k$ 步用主元 $a_{kk}^{(k)}$ 消去其下方元素：

$$
m_{ik}=\frac{a_{ik}^{(k)}}{a_{kk}^{(k)}},
\qquad
R_i\leftarrow R_i-m_{ik}R_k.
$$

消元乘子 $m_{ik}$ 会成为 LU 分解中 $L$ 的对应元素。


In [ ]:
def teaching_gaussian_elimination(A, b):
    A = np.asarray(A, dtype=float).copy()
    b = np.asarray(b, dtype=float).copy()
    n = A.shape[0]
    for k in range(n - 1):
        if abs(A[k, k]) < 1e-14:
            raise ValueError("主元为零或过小")
        for i in range(k + 1, n):
            multiplier = A[i, k] / A[k, k]
            A[i, k:] -= multiplier * A[k, k:]
            b[i] -= multiplier * b[k]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - A[i, i + 1:] @ x[i + 1:]) / A[i, i]
    return x, A, b

A = np.array([[2.0, 1.0, -1.0], [-3.0, -1.0, 2.0], [-2.0, 1.0, 2.0]])
b = np.array([8.0, -11.0, -3.0])
x, U, transformed_b = teaching_gaussian_elimination(A, b)
print("上三角矩阵 U:\n", U)
print("变换后的右端项:", transformed_b)
print("解:", x)


## 部分选主元

若主元为零，不选主元高斯消元会失败；若主元很小，消元乘子很大，舍入误差可能被放大。部分选主元在当前列中选择绝对值最大的候选主元，并交换行。


In [ ]:
def teaching_gaussian_elimination_pivoting(A, b):
    A = np.asarray(A, dtype=float).copy()
    b = np.asarray(b, dtype=float).copy()
    n = A.shape[0]
    for k in range(n - 1):
        pivot_row = k + np.argmax(np.abs(A[k:, k]))
        if abs(A[pivot_row, k]) < 1e-14:
            raise ValueError("矩阵奇异或近似奇异")
        if pivot_row != k:
            A[[k, pivot_row], :] = A[[pivot_row, k], :]
            b[[k, pivot_row]] = b[[pivot_row, k]]
        for i in range(k + 1, n):
            multiplier = A[i, k] / A[k, k]
            A[i, k:] -= multiplier * A[k, k:]
            b[i] -= multiplier * b[k]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - A[i, i + 1:] @ x[i + 1:]) / A[i, i]
    return x

A_bad = np.array([[0.0, 2.0], [1.0, 1.0]])
b_bad = np.array([2.0, 2.0])
try:
    print(teaching_gaussian_elimination(A_bad, b_bad)[0])
except ValueError as exc:
    print("不选主元失败:", exc)
print("部分选主元:", teaching_gaussian_elimination_pivoting(A_bad, b_bad))


## 小主元例子

下面的矩阵第一个主元很小。不选主元时会产生很大的消元乘子；部分选主元会先交换两行，避免这个问题。


In [ ]:
A_small = np.array([[1e-16, 1.0], [1.0, 1.0]])
x_exact = np.array([1.0, 1.0])
b_small = A_small @ x_exact

x_no_pivot = gaussian_elimination(A_small, b_small, pivoting=False, tol=0.0)
x_pivot = gaussian_elimination_partial_pivoting(A_small, b_small)
x_numpy = np.linalg.solve(A_small, b_small)

for name, x in [("不选主元", x_no_pivot), ("部分选主元", x_pivot), ("NumPy", x_numpy)]:
    print(f"{name:8s} x={x}, 残差={relative_residual(A_small, x, b_small):.3e}, 前向误差={relative_forward_error(x, x_exact):.3e}")


## 复杂度和多个右端项

一般稠密高斯消元的主要计算量是 $O(n^3)$，回代是 $O(n^2)$。如果只有一个右端项，直接消元很自然；如果有多个右端项，重复完整消元会浪费，因为系数矩阵相同。下一节的 LU 分解正是把消元过程保存下来，以便复用。


In [ ]:
rng = np.random.default_rng(2026)
A = rng.normal(size=(6, 6)) + 4.0 * np.eye(6)
B = rng.normal(size=(6, 3))
X = gaussian_elimination_partial_pivoting(A, B)
print("多个右端项相对残差:", np.linalg.norm(B - A @ X) / np.linalg.norm(B))
print("与 numpy.linalg.solve 接近:", np.linalg.norm(X - np.linalg.solve(A, B)))


## 小结

* 高斯消元把线性系统化为上三角系统。
* 消元乘子记录了每一步行变换，也是 LU 分解的来源。
* 主元为零会导致不选主元方法失败；主元过小会放大舍入误差。
* 部分选主元是一般稠密矩阵直接求解中的基本稳定策略。
